In [ ]:
# Bu dosya Jupyter/Colab notebook olarak çalıştırılacak
# Her # %%  bir hücre başlangıcını temsil eder

## [markdown]

In [ ]:
"""
# 📈 StockAI Pro — Hisse Senedi Yön Tahmini
**Yapay Zeka 2 — Final Projesi**  
Grup: Muazzez Doğru, Yunus Emre Kaymakcı, Elif Yıldırım  
Proje Kodu: C-5
"""

## HÜCRE 1: KURULUM ──────────────────────────────────────────────────────

In [ ]:
"""
Bu hücreyi Colab'da çalıştır.
VS Code'da terminal açıp:  pip install -r requirements.txt
"""
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["yfinance", "scikit-learn", "tensorflow", "plotly", "joblib"]:
    try:
        __import__(pkg.replace("-","_"))
    except ImportError:
        install(pkg)

print("✅ Tüm kütüphaneler hazır!")

## HÜCRE 2: IMPORTS ──────────────────────────────────────────────────────

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings, os, joblib
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import cross_val_score

print("✅ Import'lar tamam!")
print(f"   NumPy: {np.__version__}")
print(f"   Pandas: {pd.__version__}")
print(f"   Sklearn: ", end=""); import sklearn; print(sklearn.__version__)

## HÜCRE 3: KONFİGÜRASYON ───────────────────────────────────────────────

In [ ]:
# =============================================================================
#  BURADAN HİSSE SEÇİN
# =============================================================================
SYMBOL     = "AAPL"        # AAPL, TSLA, NVDA | THYAO.IS, GARAN.IS
START_DATE = "2019-01-01"
END_DATE   = "2024-12-31"
WINDOW     = 30            # LSTM pencere boyutu
TEST_RATIO = 0.20          # %80 train / %20 test
THRESHOLD  = 0.001         # minimum yön değişimi (gürültü filtresi)
# =============================================================================

print(f"🎯 Hedef Hisse  : {SYMBOL}")
print(f"📅 Tarih Aralığı: {START_DATE} → {END_DATE}")
print(f"🪟 LSTM Pencere : {WINDOW} gün")

## HÜCRE 4: VERİ ÇEKME ───────────────────────────────────────────────────

In [ ]:
print(f"\n📡 {SYMBOL} verisi indiriliyor...")

df_raw = yf.download(SYMBOL, start=START_DATE, end=END_DATE, auto_adjust=True)

# MultiIndex düzeltme (yfinance v0.2+)
if isinstance(df_raw.columns, pd.MultiIndex):
    df_raw.columns = df_raw.columns.get_level_values(0)

df_raw = df_raw[["Open","High","Low","Close","Volume"]].copy()
df_raw.dropna(inplace=True)

print(f"✅ Veri çekildi!")
print(f"   Satır sayısı : {len(df_raw)}")
print(f"   Tarih aralığı: {df_raw.index[0].date()} → {df_raw.index[-1].date()}")
print(f"\nİlk 5 satır:")
print(df_raw.head())

## HÜCRE 5: KEŞİFÇİ VERİ ANALİZİ (EDA) ──────────────────────────────────

In [ ]:
fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('#0a0e1a')
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.4, wspace=0.3)

# ── Fiyat Grafiği ─────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.set_facecolor('#111827')
ax1.plot(df_raw.index, df_raw["Close"], color='#00d4ff', linewidth=1.5, label='Kapanış')
ax1.fill_between(df_raw.index, df_raw["Close"], alpha=0.1, color='#00d4ff')
ax1.set_title(f'{SYMBOL} Kapanış Fiyatı', color='white', fontsize=14, fontweight='bold')
ax1.tick_params(colors='#64748b'); ax1.spines[['top','right','bottom','left']].set_color('#1e2d45')
ax1.set_ylabel('Fiyat (USD)', color='#64748b')
ax1.legend(facecolor='#1a2235', labelcolor='white')

# ── Hacim ─────────────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
ax2.set_facecolor('#111827')
ax2.bar(df_raw.index, df_raw["Volume"], color='#7c3aed', alpha=0.7, width=1)
ax2.set_title('İşlem Hacmi', color='white', fontsize=11)
ax2.tick_params(colors='#64748b'); ax2.spines[['top','right','bottom','left']].set_color('#1e2d45')
ax2.set_ylabel('Hacim', color='#64748b')

# ── Günlük Getiri Dağılımı ────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
ax3.set_facecolor('#111827')
returns = df_raw["Close"].pct_change().dropna()
ax3.hist(returns, bins=60, color='#10b981', alpha=0.8, edgecolor='#0a0e1a')
ax3.axvline(0, color='#ef4444', linestyle='--', linewidth=1.5)
ax3.set_title('Günlük Getiri Dağılımı', color='white', fontsize=11)
ax3.tick_params(colors='#64748b'); ax3.spines[['top','right','bottom','left']].set_color('#1e2d45')
ax3.set_xlabel('Getiri %', color='#64748b')

# ── Aylık Ortalama ────────────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 0])
ax4.set_facecolor('#111827')
monthly = df_raw["Close"].resample("ME").mean()
colors_m = ['#10b981' if v > monthly.mean() else '#ef4444' for v in monthly]
ax4.bar(monthly.index, monthly.values, color=colors_m, width=20)
ax4.set_title('Aylık Ortalama Fiyat', color='white', fontsize=11)
ax4.tick_params(colors='#64748b'); ax4.spines[['top','right','bottom','left']].set_color('#1e2d45')

# ── Temel İstatistikler ───────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
ax5.set_facecolor('#111827')
ax5.axis('off')
stats = df_raw["Close"].describe()
stat_text = (
    f"  TEMEL İSTATİSTİKLER\n"
    f"  {'─'*28}\n"
    f"  Ortalama   : ${stats['mean']:.2f}\n"
    f"  Min        : ${stats['min']:.2f}\n"
    f"  Max        : ${stats['max']:.2f}\n"
    f"  Std        : ${stats['std']:.2f}\n"
    f"  Sharpe*    : {returns.mean()/returns.std()*np.sqrt(252):.2f}\n"
    f"  Toplam Getiri: {((df_raw['Close'].iloc[-1]/df_raw['Close'].iloc[0])-1)*100:.1f}%"
)
ax5.text(0.05, 0.95, stat_text, transform=ax5.transAxes, color='#e2e8f0',
         fontfamily='monospace', fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='#1a2235', edgecolor='#1e2d45'))

plt.suptitle(f'{SYMBOL} — Keşifçi Veri Analizi (EDA)', color='white',
             fontsize=16, fontweight='bold', y=1.01)
plt.savefig('eda_plot.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print("✅ EDA grafiği kaydedildi → eda_plot.png")

## HÜCRE 6: TEKNİK GÖSTERGELER ───────────────────────────────────────────

In [ ]:
print("⚙️  Teknik göstergeler hesaplanıyor...")

df = df_raw.copy()
c = df["Close"]; h = df["High"]; l = df["Low"]; v = df["Volume"]

# ── Hareketli Ortalamalar ─────────────────────────────────────────────────────
df["MA10"]  = c.rolling(10).mean()
df["MA20"]  = c.rolling(20).mean()
df["MA50"]  = c.rolling(50).mean()
df["EMA10"] = c.ewm(span=10, adjust=False).mean()
df["EMA20"] = c.ewm(span=20, adjust=False).mean()

# ── RSI ───────────────────────────────────────────────────────────────────────
delta = c.diff()
gain  = delta.clip(lower=0).ewm(com=13, min_periods=14).mean()
loss  = (-delta.clip(upper=0)).ewm(com=13, min_periods=14).mean()
df["RSI"] = 100 - (100 / (1 + gain/(loss+1e-9)))

# ── MACD ──────────────────────────────────────────────────────────────────────
ema12 = c.ewm(span=12, adjust=False).mean()
ema26 = c.ewm(span=26, adjust=False).mean()
df["MACD"]        = ema12 - ema26
df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
df["MACD_Hist"]   = df["MACD"] - df["MACD_Signal"]

# ── Bollinger Bands ───────────────────────────────────────────────────────────
sma20 = c.rolling(20).mean()
std20 = c.rolling(20).std()
df["BB_Upper"] = sma20 + 2*std20
df["BB_Lower"] = sma20 - 2*std20
df["BB_Width"] = (df["BB_Upper"]-df["BB_Lower"]) / (sma20+1e-9)
df["BB_Pos"]   = (c-df["BB_Lower"]) / (df["BB_Upper"]-df["BB_Lower"]+1e-9)

# ── Stochastic ────────────────────────────────────────────────────────────────
low14  = l.rolling(14).min()
high14 = h.rolling(14).max()
df["Stoch_K"] = 100*(c-low14)/(high14-low14+1e-9)
df["Stoch_D"] = df["Stoch_K"].rolling(3).mean()

# ── ATR ───────────────────────────────────────────────────────────────────────
tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
df["ATR"] = tr.rolling(14).mean()

# ── OBV ───────────────────────────────────────────────────────────────────────
obv = [0]
for i in range(1, len(df)):
    obv.append(obv[-1]+v.iloc[i] if c.iloc[i]>c.iloc[i-1] else
               obv[-1]-v.iloc[i] if c.iloc[i]<c.iloc[i-1] else obv[-1])
df["OBV"] = obv

# ── Diğer Özellikler ──────────────────────────────────────────────────────────
df["Return"]       = c.pct_change()
df["Volatility"]   = df["Return"].rolling(10).std()
df["Momentum"]     = c - c.shift(10)
df["Price_vs_MA"]  = c/df["MA20"] - 1
df["Trend"]        = (c > df["MA50"]).astype(int)
df["Volume_MA"]    = v.rolling(20).mean()
df["Volume_Ratio"] = v/(df["Volume_MA"]+1e-9)

df.dropna(inplace=True)

print(f"✅ {len(df.columns)} özellik hesaplandı!")
print(f"   Veri satır sayısı: {len(df)}")
print(f"\nÖzellikler: {', '.join([c for c in df.columns if c not in ['Open','High','Low','Close','Volume']])}")

## HÜCRE 7: HEDEF DEĞİŞKEN ───────────────────────────────────────────────

In [ ]:
# Ertesi gün fiyat yönü: 1=Yukarı, 0=Aşağı
future_return = df["Close"].shift(-1) / df["Close"] - 1
df["Target"] = (future_return > THRESHOLD).astype(int)
df = df.iloc[:-1]  # son satır NaN hedef

up   = df["Target"].sum()
down = len(df["Target"]) - up
print(f"📊 Sınıf Dağılımı:")
print(f"   ↑ Yukarı (1): {up}  ({up/len(df)*100:.1f}%)")
print(f"   ↓ Aşağı  (0): {down} ({down/len(df)*100:.1f}%)")

## HÜCRE 8: VERİ HAZIRLAMA ───────────────────────────────────────────────

In [ ]:
FEATURES = [
    "Close","MA10","MA20","MA50","EMA10","EMA20",
    "RSI","MACD","MACD_Signal","MACD_Hist",
    "BB_Width","BB_Pos","Stoch_K","Stoch_D",
    "ATR","Return","Volatility","Momentum",
    "Price_vs_MA","Trend","Volume_Ratio"
]

# Random Forest için
scaler_rf = StandardScaler()
X_rf = scaler_rf.fit_transform(df[FEATURES])
y    = df["Target"].values

split = int(len(X_rf) * (1 - TEST_RATIO))
X_train_rf, X_test_rf = X_rf[:split], X_rf[split:]
y_train,    y_test    = y[:split],    y[split:]

# LSTM için - Sliding Window
scaler_lstm = MinMaxScaler()
X_scaled = scaler_lstm.fit_transform(df[FEATURES])

X_lstm, y_lstm = [], []
for i in range(WINDOW, len(X_scaled)-1):
    X_lstm.append(X_scaled[i-WINDOW:i])
    y_lstm.append(y[i])
X_lstm, y_lstm = np.array(X_lstm), np.array(y_lstm)

split_lstm = int(len(X_lstm)*(1-TEST_RATIO))
X_train_lstm, X_test_lstm = X_lstm[:split_lstm], X_lstm[split_lstm:]
y_train_lstm, y_test_lstm = y_lstm[:split_lstm], y_lstm[split_lstm:]

print(f"✅ Veri hazırlandı!")
print(f"\n── Random Forest ──────────────────")
print(f"   Train: {X_train_rf.shape} | Test: {X_test_rf.shape}")
print(f"\n── LSTM ───────────────────────────")
print(f"   Train: {X_train_lstm.shape} | Test: {X_test_lstm.shape}")
print(f"   (pencere={WINDOW}, özellik={len(FEATURES)})")

## HÜCRE 9: RANDOM FOREST EĞİTİMİ ────────────────────────────────────────

In [ ]:
print("🌲 Random Forest eğitiliyor...")

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_rf, y_train)

# Değerlendirme
rf_pred  = rf_model.predict(X_test_rf)
rf_prob  = rf_model.predict_proba(X_test_rf)[:,1]
rf_acc   = accuracy_score(y_test, rf_pred)
rf_auc   = roc_auc_score(y_test, rf_prob)

print(f"\n{'═'*45}")
print(f"  RANDOM FOREST SONUÇLARI")
print(f"{'═'*45}")
print(f"  Doğruluk  : {rf_acc*100:.2f}%")
print(f"  AUC-ROC   : {rf_auc:.4f}")
print(f"{'─'*45}")
print(classification_report(y_test, rf_pred, target_names=["AŞAĞI","YUKARI"]))

## HÜCRE 10: LSTM EĞİTİMİ ─────────────────────────────────────────────────

In [ ]:
print("🧠 LSTM modeli kuruluyor...")

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import (LSTM, Bidirectional, Dense,
                                         Dropout, BatchNormalization)
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    print(f"   TensorFlow {tf.__version__} ✅")

    lstm_model = Sequential([
        Bidirectional(LSTM(128, return_sequences=True),
                      input_shape=(WINDOW, len(FEATURES))),
        BatchNormalization(),
        Dropout(0.3),
        LSTM(64, return_sequences=True),
        BatchNormalization(),
        Dropout(0.3),
        LSTM(32, return_sequences=False),
        BatchNormalization(),
        Dropout(0.2),
        Dense(64, activation="relu"),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(1,  activation="sigmoid")
    ])

    lstm_model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    lstm_model.summary()

    callbacks = [
        EarlyStopping(patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(factor=0.5, patience=4, min_lr=1e-6, verbose=1)
    ]

    history = lstm_model.fit(
        X_train_lstm, y_train_lstm,
        validation_data=(X_test_lstm, y_test_lstm),
        epochs=50,
        batch_size=32,
        callbacks=callbacks,
        verbose=1
    )

    lstm_pred = (lstm_model.predict(X_test_lstm, verbose=0).flatten() > 0.5).astype(int)
    lstm_prob = lstm_model.predict(X_test_lstm, verbose=0).flatten()
    lstm_acc  = accuracy_score(y_test_lstm, lstm_pred)
    lstm_auc  = roc_auc_score(y_test_lstm, lstm_prob)

    print(f"\n{'═'*45}")
    print(f"  LSTM SONUÇLARI")
    print(f"{'═'*45}")
    print(f"  Doğruluk  : {lstm_acc*100:.2f}%")
    print(f"  AUC-ROC   : {lstm_auc:.4f}")
    print(f"{'─'*45}")
    print(classification_report(y_test_lstm, lstm_pred, target_names=["AŞAĞI","YUKARI"]))
    LSTM_AVAILABLE = True

except ImportError:
    print("⚠️  TensorFlow kurulu değil → sadece RF modeli kullanılacak")
    LSTM_AVAILABLE = False
    lstm_acc, lstm_auc = 0, 0

## HÜCRE 11: GRAFİKLER & DEĞERLENDİRME ───────────────────────────────────

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor('#0a0e1a')
bg = '#111827'; text_c = '#e2e8f0'; grid_c = '#1e2d45'

def style_ax(ax, title):
    ax.set_facecolor(bg)
    ax.set_title(title, color=text_c, fontsize=12, fontweight='bold')
    ax.tick_params(colors='#64748b')
    for spine in ax.spines.values():
        spine.set_color(grid_c)
    ax.grid(color=grid_c, linewidth=0.5, alpha=0.7)

# ── 1. Feature Importance ─────────────────────────────────────────────────────
ax = axes[0,0]
style_ax(ax, "Feature Importance (RF)")
fi = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values()
colors_fi = plt.cm.plasma(np.linspace(0.3, 0.9, len(fi)))
ax.barh(fi.index, fi.values, color=colors_fi)
ax.set_xlabel("Önem", color='#64748b')

# ── 2. ROC Eğrisi ─────────────────────────────────────────────────────────────
ax = axes[0,1]
style_ax(ax, "ROC Eğrisi")
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_prob)
ax.plot(fpr_rf, tpr_rf, color='#00d4ff', lw=2, label=f"RF  AUC={rf_auc:.3f}")
if LSTM_AVAILABLE:
    fpr_l, tpr_l, _ = roc_curve(y_test_lstm, lstm_prob)
    ax.plot(fpr_l, tpr_l, color='#7c3aed', lw=2, label=f"LSTM AUC={lstm_auc:.3f}")
ax.plot([0,1],[0,1],'--',color='#64748b',lw=1)
ax.fill_between(fpr_rf, tpr_rf, alpha=0.1, color='#00d4ff')
ax.set_xlabel("FPR", color='#64748b'); ax.set_ylabel("TPR", color='#64748b')
legend = ax.legend(facecolor='#1a2235', labelcolor=text_c)

# ── 3. Confusion Matrix ───────────────────────────────────────────────────────
ax = axes[0,2]
style_ax(ax, "Confusion Matrix (RF)")
cm = confusion_matrix(y_test, rf_pred)
im = ax.imshow(cm, cmap='Blues', aspect='auto')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                color=text_c, fontsize=16, fontweight='bold')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(["AŞAĞI","YUKARI"], color=text_c)
ax.set_yticklabels(["AŞAĞI","YUKARI"], color=text_c)
ax.set_xlabel("Tahmin", color='#64748b'); ax.set_ylabel("Gerçek", color='#64748b')

# ── 4. Fiyat + Tahmin Sinyalleri ──────────────────────────────────────────────
ax = axes[1,0]
style_ax(ax, "Fiyat + Model Sinyalleri (RF)")
# rf_pred ve test_prices aynı uzunlukta olmalı
n = min(len(rf_pred), len(df["Close"].iloc[split:]))
test_prices = df["Close"].iloc[split:].values[:n]
rf_pred_plot = rf_pred[:n]
test_dates  = range(n)
ax.plot(test_dates, test_prices, color='#94a3b8', lw=1.2, label='Fiyat', zorder=1)
buy_idx  = np.where(rf_pred_plot == 1)[0]
sell_idx = np.where(rf_pred_plot == 0)[0]
if len(buy_idx):
    ax.scatter(buy_idx, test_prices[buy_idx], color='#10b981',
               marker='^', s=30, label='AL', zorder=3, alpha=0.8)
if len(sell_idx):
    ax.scatter(sell_idx, test_prices[sell_idx], color='#ef4444',
               marker='v', s=30, label='SAT', zorder=3, alpha=0.8)
legend2 = ax.legend(facecolor='#1a2235', labelcolor=text_c)
ax.set_xlabel("Gün", color='#64748b'); ax.set_ylabel("Fiyat ($)", color='#64748b')

# ── 5. LSTM Eğitim Eğrisi ─────────────────────────────────────────────────────
ax = axes[1,1]
style_ax(ax, "LSTM Eğitim Eğrisi")
if LSTM_AVAILABLE and 'history' in dir():
    ax.plot(history.history['accuracy'], color='#00d4ff', lw=2, label='Train Acc')
    ax.plot(history.history['val_accuracy'], color='#7c3aed', lw=2, label='Val Acc')
    ax.plot(history.history['loss'], color='#10b981', lw=1.5, linestyle='--', label='Train Loss')
    ax.plot(history.history['val_loss'], color='#ef4444', lw=1.5, linestyle='--', label='Val Loss')
    legend3 = ax.legend(facecolor='#1a2235', labelcolor=text_c)
    ax.set_xlabel("Epoch", color='#64748b')
else:
    ax.text(0.5, 0.5, "LSTM\nMevcut Değil", ha='center', va='center',
            color='#64748b', fontsize=14, transform=ax.transAxes)

# ── 6. Model Karşılaştırma ────────────────────────────────────────────────────
ax = axes[1,2]
style_ax(ax, "Model Karşılaştırma")
models = ["Random Forest"]
accs   = [rf_acc*100]
aucs   = [rf_auc]
if LSTM_AVAILABLE:
    models.append("LSTM")
    accs.append(lstm_acc*100)
    aucs.append(lstm_auc)
x = np.arange(len(models))
w = 0.35
b1 = ax.bar(x-w/2, accs, w, label="Doğruluk %", color='#00d4ff', alpha=0.85)
b2 = ax.bar(x+w/2, [a*100 for a in aucs], w, label="AUC×100", color='#7c3aed', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(models, color=text_c)
ax.set_ylim(0, 110)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{bar.get_height():.1f}', ha='center', color=text_c, fontsize=9)
legend4 = ax.legend(facecolor='#1a2235', labelcolor=text_c)

plt.suptitle(f'{SYMBOL} — Model Değerlendirme Grafikleri',
             color=text_c, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print("✅ Model değerlendirme grafiği kaydedildi → model_evaluation.png")

## HÜCRE 12: BACKTEST SİMÜLASYONU ────────────────────────────────────────

In [ ]:
print("📊 Backtest simülasyonu çalışıyor...")

initial_capital = 10000  # $10,000 başlangıç
capital    = initial_capital
position   = 0
portfolio  = []
buy_hold   = []
bh_capital = initial_capital

n_bt = min(len(rf_pred), len(df["Close"].iloc[split:]))
test_prices_bt = df["Close"].iloc[split:].values[:n_bt]
rf_pred = rf_pred[:n_bt]

for i, (price, signal) in enumerate(zip(test_prices_bt, rf_pred)):
    # Buy & Hold
    if i == 0:
        bh_shares = bh_capital / price
    bh_capital = bh_shares * price

    # Model Stratejisi
    if signal == 1 and position == 0:      # AL
        position = capital / price
        capital  = 0
    elif signal == 0 and position > 0:     # SAT
        capital  = position * price
        position = 0

    current_val = capital + position * price
    portfolio.append(current_val)
    buy_hold.append(bh_capital)

# Son pozisyonu kapat
if position > 0:
    portfolio[-1] = position * test_prices_bt[-1]

final_model    = portfolio[-1]
final_bh       = buy_hold[-1]
model_return   = (final_model - initial_capital) / initial_capital * 100
bh_return      = (final_bh   - initial_capital) / initial_capital * 100

print(f"\n{'═'*45}")
print(f"  BACKTEST SONUÇLARI ({len(test_prices_bt)} işlem günü)")
print(f"{'═'*45}")
print(f"  Başlangıç Sermaye  : ${initial_capital:,.0f}")
print(f"  Model Stratejisi   : ${final_model:,.0f}  ({model_return:+.1f}%)")
print(f"  Buy & Hold         : ${final_bh:,.0f}  ({bh_return:+.1f}%)")
print(f"  Fark               : ${final_model-final_bh:+,.0f}")
print(f"{'═'*45}")

# Backtest grafiği
fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('#0a0e1a'); ax.set_facecolor('#111827')
ax.plot(portfolio, color='#00d4ff', lw=2, label=f'Model ({model_return:+.1f}%)')
ax.plot(buy_hold,  color='#7c3aed', lw=2, linestyle='--', label=f'Buy&Hold ({bh_return:+.1f}%)')
ax.fill_between(range(len(portfolio)), portfolio, buy_hold,
                where=[p>b for p,b in zip(portfolio,buy_hold)],
                color='#10b981', alpha=0.15, label='Model > B&H')
ax.fill_between(range(len(portfolio)), portfolio, buy_hold,
                where=[p<=b for p,b in zip(portfolio,buy_hold)],
                color='#ef4444', alpha=0.15)
ax.axhline(initial_capital, color='#64748b', lw=1, linestyle=':')
ax.set_title(f'{SYMBOL} Backtest — Model vs Buy&Hold', color='white', fontsize=14, fontweight='bold')
ax.tick_params(colors='#64748b')
for spine in ax.spines.values(): spine.set_color('#1e2d45')
ax.grid(color='#1e2d45', linewidth=0.5, alpha=0.7)
ax.set_xlabel("Gün", color='#64748b'); ax.set_ylabel("Portföy ($)", color='#64748b')
legend = ax.legend(facecolor='#1a2235', labelcolor='white')
plt.tight_layout()
plt.savefig('backtest.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()

## HÜCRE 13: MODELLERİ KAYDET ────────────────────────────────────────────

In [ ]:
os.makedirs("models", exist_ok=True)

joblib.dump(rf_model,    "models/rf_model.pkl")
joblib.dump(scaler_rf,   "models/scaler_rf.pkl")
joblib.dump(scaler_lstm, "models/scaler_lstm.pkl")
joblib.dump(FEATURES,    "models/features.pkl")

if LSTM_AVAILABLE:
    lstm_model.save("models/lstm_model.keras")
    print("✅ LSTM modeli kaydedildi → models/lstm_model.keras")

print("✅ Tüm modeller kaydedildi:")
print("   📁 models/rf_model.pkl")
print("   📁 models/scaler_rf.pkl")
print("   📁 models/scaler_lstm.pkl")
print("   📁 models/features.pkl")
print("\n🎉 Notebook tamamlandı! Streamlit arayüzünü çalıştırmak için:")
print("   streamlit run app.py")